### Importations et Dépendances

In [17]:
import os
import scipy.io
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import normalized_mutual_info_score
import os
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

### Configuration des chemins et hyperparamètres

In [20]:
# --- CONFIGURATION ---
DATA_ROOT = r"C:\Users\Home\Documents\M2\Projet_PPD\data"
SAVE_ROOT = r"C:\Users\Home\Documents\M2\Projet_PPD\output\WeTe"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATASETS = ["20NG", "IMDB", "YahooAnswer", "AGNews"]
K_VALUES = [50, 100]

# Paramètres pour le TD
ULTRA_CONFIG = {
    "20NG": {"lr": 0.001, "kl_beta": 0.08, "div_weight": 5000.0, "top_filter": 400},
    "IMDB": {"lr": 0.0008, "kl_beta": 0.05, "div_weight": 4000.0, "top_filter": 150},
    "YahooAnswer": {"lr": 0.001, "kl_beta": 0.1, "div_weight": 3500.0, "top_filter": 300},
    "AGNews": {"lr": 0.001, "kl_beta": 0.1, "div_weight": 4500.0, "top_filter": 200},
    "default": {"lr": 0.001, "kl_beta": 0.1, "div_weight": 3000.0, "top_filter": 200}
}

### Fonctions de chargement des données

In [3]:
def load_dataset_(name):
    path = os.path.join(DATA_ROOT, name)
    train = sp.load_npz(os.path.join(path, "train_bow.npz")).toarray()
    test = sp.load_npz(os.path.join(path, "test_bow.npz")).toarray()
    vocab = [w.strip() for w in open(os.path.join(path, "vocab.txt"), encoding="utf-8")]
    test_labels = np.loadtxt(os.path.join(path, "test_labels.txt"), dtype=int)
    return train, test, vocab, test_labels


SEED = 42 

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)


### Définition de l'architecture du modèle WeTe

In [4]:
class WeTe_Model(nn.Module):
    def __init__(self, vocab_size, num_topics, rho_size=300, t_hidden_size=800, dropout=0.5):
        super().__init__()
        self.num_topics = num_topics
        self.rho = nn.Linear(rho_size, vocab_size, bias=False)
        self.alpha = nn.Linear(rho_size, num_topics, bias=False)
        
        nn.init.orthogonal_(self.rho.weight)
        nn.init.orthogonal_(self.alpha.weight)
        
        self.encoder = nn.Sequential(
            nn.Linear(vocab_size, t_hidden_size),
            nn.BatchNorm1d(t_hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(t_hidden_size, t_hidden_size),
            nn.BatchNorm1d(t_hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.fc_mu = nn.Linear(t_hidden_size, num_topics)
        self.fc_logvar = nn.Linear(t_hidden_size, num_topics)

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def get_beta(self, temp=0.1):
        logit_beta = self.alpha(self.rho.weight) 
        if not self.training:
            # Stratégie WeTe : Masquage Top-K pour l'exclusivité
            vals, idx = torch.topk(logit_beta, k=50, dim=1)
            mask = torch.zeros_like(logit_beta).fill_(-1e9)
            mask.scatter_(1, idx, vals)
            logit_beta = mask
        return F.softmax(logit_beta / temp, dim=0).t() 

    def get_diversity_loss(self):
        a = self.alpha.weight
        norm_a = F.normalize(a, p=2, dim=1)
        correlation_matrix = torch.mm(norm_a, norm_a.t())
        identity = torch.eye(self.num_topics).to(DEVICE)
        return F.mse_loss(correlation_matrix, identity)

    def forward(self, x, kl_w, div_w):
        x_norm = x / (x.sum(1, keepdim=True) + 1e-10)
        enc = self.encoder(x_norm)
        mu, logvar = self.fc_mu(enc), self.fc_logvar(enc)
        z = self.reparameterize(mu, logvar)
        theta = F.softmax(z, dim=-1)
        beta = self.get_beta()
        recon = torch.mm(theta, beta)
        recon_loss = -(x * (recon + 1e-10).log()).sum(1).mean()
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=-1).mean()
        div_loss = self.get_diversity_loss()
        return recon_loss + (kl_w * kl_loss) + (div_w * div_loss), recon_loss, kl_loss

### Fonction d'entraînement

In [5]:
def train_refinement_wete(dataset_name, K):
    train_bow, _, vocab, _ = load_dataset_(dataset_name)
    
    config = ULTRA_CONFIG.get(dataset_name, ULTRA_CONFIG["default"])
    
    # Prétraitement / Filtrage
    filter_val = 500 if dataset_name == "20NG" else config.get("top_filter", 200)
    word_counts = train_bow.sum(axis=0)
    top_indices = np.argsort(word_counts)[-filter_val:]
    train_bow_clean = train_bow.copy()
    train_bow_clean[:, top_indices] = 0
    
    # Instanciation du modèle WeTe
    model = WeTe_Model(train_bow.shape[1], K).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=10, factor=0.5)
    
    loader = DataLoader(TensorDataset(torch.from_numpy(train_bow_clean).float()), batch_size=512, shuffle=True)
    
    for epoch in tqdm(range(150), desc=f"WeTe Training {dataset_name} K={K}", leave=False):
        model.train()
        kl_w = min(config["kl_beta"], epoch / 50.0)
        
        total_loss = 0
        for batch in loader:
            x = batch[0].to(DEVICE)
            optimizer.zero_grad()
            loss, recon, kl = model(x, kl_w, config["div_weight"] * 2.0)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            total_loss += loss.item()
            
        scheduler.step(total_loss)
            
    return model

### Fonctions utilitaires de calcul des métriques

In [6]:
def get_metrics(beta, theta, y_test, vocab):
    # Topic Diversity
    num_topics = beta.shape[0]
    list_w = []
    for k in range(num_topics):
        idx = beta[k].argsort()[-10:][::-1]
        list_w.extend([vocab[i] for i in idx])
    td = len(set(list_w)) / len(list_w)
    
    # Clustering (Purity & NMI)
    preds = theta.argmax(1)
    nmi = normalized_mutual_info_score(y_test, preds)
    
    # Purity
    y_voted = np.zeros(y_test.shape)
    labels = np.unique(y_test)
    for cluster in np.unique(preds):
        hist, _ = np.histogram(y_test[preds == cluster], bins=np.append(labels, labels.max() + 1))
        y_voted[preds == cluster] = labels[np.argmax(hist)]
    purity = np.mean(y_voted == y_test)
    
    return td, nmi, purity

### Entrainement du modèle et calcule des métriques (TD, Purity, NMI)

In [16]:
SAVE_PATH = os.path.join(SAVE_ROOT, "wete_results.csv")

# 1. Charger les résultats existants si c'est le cas
if os.path.exists(SAVE_PATH):
    df_existing = pd.read_csv(SAVE_PATH)
    all_results_fast = df_existing.to_dict('records')
    print(f" {len(all_results_fast)} résultats chargés depuis le checkpoint.")
else:
    all_results_fast = []

# On boucle sur vos datasets et valeurs de K
for dataset_name in DATASETS:
    print(f"\n" + "="*50)
    print(f"DATASET : {dataset_name}")
    print("="*50)
    
    # Charger les données 
    train_bow, test_bow, vocab, y_test = load_dataset_(dataset_name)
    
    for K in K_VALUES:
        already_done = any(r['Dataset'] == dataset_name and r['K'] == K for r in all_results_fast)
        
        if already_done:
            print(f"WeTe | K={K} | Déjà calculé, on passe au suivant.")
            continue
        
    
        print(f"\n WeTe | K={K} | Entraînement...")
        
        # 2. Entraînement du modèle WeTe
        model = train_refinement_wete(dataset_name, K)
        model.eval()
        
        # 3. Inférence
        with torch.no_grad():
            beta = model.get_beta(temp=0.1).cpu().numpy()
            test_tensor = torch.from_numpy(test_bow).float().to(DEVICE)
            theta = F.softmax(model.fc_mu(model.encoder(test_tensor)), dim=-1).cpu().numpy()
            
        # 4. Calcul des métriques
        print(f"Calcul des scores (TD, Purity, NMI)...")
        td, nmi, pur = get_metrics(beta, theta, y_test, vocab)
        
        # 5. Stockage et Sauvegarde immédiate
        res = {
            "Dataset": dataset_name,
            "K": K,
            "TD": round(td, 3),
            "Purity": round(pur, 3),
            "NMI": round(nmi, 3),
        }
        all_results_fast.append(res)
        
        pd.DataFrame(all_results_fast).to_csv(SAVE_PATH, index=False)
        
        print(f" K={K} | TD: {res['TD']} | Pur: {res['Purity']} | NMI: {res['NMI']}")

# --- AFFICHAGE DU TABLEAU DE SYNTHÈSE ---
df_wete_fast = pd.DataFrame(all_results_fast)
print("\n" + "#"*60)
print("          RÉSULTATS DE REPRODUCTION WeTe")
print("#"*60)
print(df_wete_fast.to_string(index=False))

 8 résultats chargés depuis le checkpoint.

DATASET : 20NG
WeTe | K=50 | Déjà calculé, on passe au suivant.
WeTe | K=100 | Déjà calculé, on passe au suivant.

DATASET : IMDB
WeTe | K=50 | Déjà calculé, on passe au suivant.
WeTe | K=100 | Déjà calculé, on passe au suivant.

DATASET : YahooAnswer
WeTe | K=50 | Déjà calculé, on passe au suivant.
WeTe | K=100 | Déjà calculé, on passe au suivant.

DATASET : AGNews
WeTe | K=50 | Déjà calculé, on passe au suivant.
WeTe | K=100 | Déjà calculé, on passe au suivant.

############################################################
          RÉSULTATS DE REPRODUCTION WeTe
############################################################
    Dataset   K    TD  Purity   NMI
       20NG  50 0.934   0.513 0.464
       20NG 100 0.704   0.542 0.493
       IMDB  50 0.974   0.676 0.046
       IMDB 100 0.788   0.697 0.055
YahooAnswer  50 0.866   0.470 0.261
YahooAnswer 100 0.607   0.517 0.285
     AGNews  50 0.830   0.776 0.345
     AGNews 100 0.593   0.792 0.358


### Calcul de la métrique Coherence C_v

In [18]:
# Nom du fichier de sauvegarde pour la cohérence
CV_SAVE_PATH = os.path.join(SAVE_ROOT,"wete_cv.csv")
TOPN = 15

# 1. Charger les résultats C_V existants
if os.path.exists(CV_SAVE_PATH):
    df_cv_existing = pd.read_csv(CV_SAVE_PATH)
    wete_cv_results = df_cv_existing.to_dict('records')
    print(f"{len(wete_cv_results)} scores C_V chargés depuis le checkpoint.")
else:
    wete_cv_results = []

print("CALCUL C_V PALMETTO")

for dataset_name in DATASETS:
    # Vérification si au moins un K pour ce dataset manque avant de charger le corpus
    needed_ks = [k for k in K_VALUES if not any(r['Dataset'] == dataset_name and r['K'] == k for r in wete_cv_results)]
    
    if not needed_ks:
        print(f"{dataset_name} | Tous les K sont déjà calculés pour la cohérence.")
        continue

    print(f"\n" + "="*50)
    print(f" TRAITEMENT COMPLET : {dataset_name}")
    print("="*50)
    
    # Chargement et reconstruction du corpus (Full)
    train_bow, _, vocab_list, _ = load_dataset_(dataset_name)
    print(f"Reconstruction de tous les textes ({train_bow.shape[0]} docs)...")
    
    texts = []
    for i in range(train_bow.shape[0]):
        idx = train_bow[i].nonzero()[0]
        if len(idx) > 1:
            texts.append([vocab_list[w] for w in idx])
    
    dictionary = Dictionary(texts)
    
    for K in K_VALUES:
        # SKIP si déjà dans le CSV
        if any(r['Dataset'] == dataset_name and r['K'] == K for r in wete_cv_results):
            print(f" K={K} déjà présent, skip.")
            continue
            
        print(f"\n WeTe | K={K} | Calcul C_V en cours...")
        
        # Récupération du modèle
        model = train_refinement_wete(dataset_name, K)
        model.eval()
        
        with torch.no_grad():
            beta = model.get_beta(temp=0.1).cpu().numpy()
        
        # Extraction des Topic IDs
        topics_ids = []
        for k in range(K):
            idx = beta[k].argsort()[-TOPN:][::-1]
            t_words = [vocab_list[i] for i in idx]
            ids = [dictionary.token2id[w] for w in t_words if w in dictionary.token2id]
            if len(ids) >= 2:
                topics_ids.append(ids)
        
        # Calcul Gensim
        cv_score = 0.0
        if topics_ids:
            try:
                cm = CoherenceModel(topics=topics_ids, texts=texts, dictionary=dictionary, coherence='c_v')
                cv_score = cm.get_coherence()
            except Exception as e:
                print(f"Erreur Gensim : {e}")
        
        # Stockage et sauvegarde
        res_cv = {"Dataset": dataset_name, "K": K, "C_V": round(cv_score, 4)}
        wete_cv_results.append(res_cv)
        pd.DataFrame(wete_cv_results).to_csv(CV_SAVE_PATH, index=False)
        
        print(f"{dataset_name} K={K} | C_V = {cv_score:.4f}")

# Affichage final
df_cv_final = pd.DataFrame(wete_cv_results)
print("\n" + "#"*50 + "\n RÉCAPITULATIF COHÉRENCE WeTe\n" + "#"*50)
print(df_cv_final.pivot(index="Dataset", columns="K", values="C_V"))

CALCUL C_V PALMETTO

 TRAITEMENT COMPLET : 20NG
Reconstruction de tous les textes (11314 docs)...

 WeTe | K=50 | Calcul C_V en cours...


WeTe Training 20NG K=50:   0%|          | 0/150 [00:00<?, ?it/s]

20NG K=50 | C_V = 0.4086

 WeTe | K=100 | Calcul C_V en cours...


WeTe Training 20NG K=100:   0%|          | 0/150 [00:00<?, ?it/s]

20NG K=100 | C_V = 0.2996

 TRAITEMENT COMPLET : IMDB
Reconstruction de tous les textes (25000 docs)...

 WeTe | K=50 | Calcul C_V en cours...


...

YahooAnswer K=100 | C_V = 0.3438

 TRAITEMENT COMPLET : AGNews
Reconstruction de tous les textes (10000 docs)...

 WeTe | K=50 | Calcul C_V en cours...


WeTe Training AGNews K=50:   0%|          | 0/150 [00:00<?, ?it/s]

AGNews K=50 | C_V = 0.3366

 WeTe | K=100 | Calcul C_V en cours...


WeTe Training AGNews K=100:   0%|          | 0/150 [00:00<?, ?it/s]

AGNews K=100 | C_V = 0.3793

##################################################
 RÉCAPITULATIF COHÉRENCE WeTe
##################################################
K               50      100
Dataset                    
20NG         0.4086  0.2996
AGNews       0.3366  0.3793
IMDB         0.3858  0.3493
YahooAnswer  0.3604  0.3438


### Comparaison des résultats avec les résultats du papier

In [19]:
import pandas as pd

# 1. Données du papier 
data_paper = {
    'Dataset': ['20NG', '20NG', 'IMDB', 'IMDB', 'YahooAnswer', 'YahooAnswer', 'AGNews', 'AGNews'],
    'K': [50, 100, 50, 100, 50, 100, 50, 100],
    'CV_pap': [0.383, 0.352, 0.368, 0.293, 0.367, 0.353, 0.383, 0.363],
    'TD_pap': [0.949, 0.742, 0.931, 0.638, 0.878, 0.544, 0.945, 0.827],
    'Purity_pap': [0.268, 0.338, 0.587, 0.589, 0.389, 0.444, 0.641, 0.699],
    'NMI_pap': [0.304, 0.348, 0.031, 0.025, 0.252, 0.269, 0.268, 0.271]
}

# 2. Données de la reproduction 
data_repro = {
    'Dataset': ['20NG', '20NG', 'AGNews', 'AGNews', 'IMDB', 'IMDB', 'YahooAnswer', 'YahooAnswer'],
    'K': [50, 100, 50, 100, 50, 100, 50, 100],
    'CV_rep': [0.409, 0.300, 0.337, 0.379, 0.386, 0.349, 0.360, 0.344],
    'TD_rep': [0.934, 0.704, 0.830, 0.593, 0.974, 0.788, 0.866, 0.607],
    'Purity_rep': [0.513, 0.542, 0.776, 0.792, 0.676, 0.697, 0.470, 0.517],
    'NMI_rep': [0.464, 0.493, 0.345, 0.358, 0.046, 0.055, 0.261, 0.285]
}

df_pap = pd.DataFrame(data_paper)
df_rep = pd.DataFrame(data_repro)

# Fusion des deux tableaux
df_compare = pd.merge(df_pap, df_rep, on=['Dataset', 'K'])

# 3. Calcul des écarts (Reproduction - Papier)
df_compare['Diff_CV'] = (df_compare['CV_rep'] - df_compare['CV_pap']).round(3)
df_compare['Diff_TD'] = (df_compare['TD_rep'] - df_compare['TD_pap']).round(3)
df_compare['Diff_Purity'] = (df_compare['Purity_rep'] - df_compare['Purity_pap']).round(3)
df_compare['Diff_NMI'] = (df_compare['NMI_rep'] - df_compare['NMI_pap']).round(3)

# Sélection et organisation des colonnes
final_cols = [
    'Dataset', 'K', 
    'CV_pap', 'CV_rep', 'Diff_CV',
    'TD_pap', 'TD_rep', 'Diff_TD',
    'Purity_pap', 'Purity_rep', 'Diff_Purity',
    'NMI_pap', 'NMI_rep', 'Diff_NMI'
]

print("="*120)
print(f"{'COMPARAISON FINALE WeTe : RÉSULTATS ORIGINAUX VS REPRODUCTION':^120}")
print("="*120)
print(df_compare[final_cols].to_string(index=False))

                             COMPARAISON FINALE WeTe : RÉSULTATS ORIGINAUX VS REPRODUCTION                              
    Dataset   K  CV_pap  CV_rep  Diff_CV  TD_pap  TD_rep  Diff_TD  Purity_pap  Purity_rep  Diff_Purity  NMI_pap  NMI_rep  Diff_NMI
       20NG  50   0.383   0.409    0.026   0.949   0.934   -0.015       0.268       0.513        0.245    0.304    0.464     0.160
       20NG 100   0.352   0.300   -0.052   0.742   0.704   -0.038       0.338       0.542        0.204    0.348    0.493     0.145
       IMDB  50   0.368   0.386    0.018   0.931   0.974    0.043       0.587       0.676        0.089    0.031    0.046     0.015
       IMDB 100   0.293   0.349    0.056   0.638   0.788    0.150       0.589       0.697        0.108    0.025    0.055     0.030
YahooAnswer  50   0.367   0.360   -0.007   0.878   0.866   -0.012       0.389       0.470        0.081    0.252    0.261     0.009
YahooAnswer 100   0.353   0.344   -0.009   0.544   0.607    0.063       0.444       0.517    